# The colour-MNIST judge, qualitatively

Loads the digit/fg/bg classifier the evaluation scores samples with and shows val images
captioned with their true and predicted labels. Captions in red are misclassified in at
least one factor.

In [ ]:
import torch
from torch.utils.data import DataLoader

from dataset_loaders import build_dataset
from dataset_loaders.colour_mnist import BG_NAMES, FG_NAMES
from utils.checkpoints import load_classifier_from_path
from utils.reproducibility import resolve_device, seed_everything
from utils.visualisation import plot_captioned_images
from utils.wandb_utils import download_artifact

device = resolve_device()
seed_everything(0)

## Load the classifier and the val split

In [ ]:
CLASSIFIER_ARTIFACT = "digit_classifier_colour_mnist_uniform"
DATASET = "colour_mnist_uniform"

path, resolved = download_artifact(CLASSIFIER_ARTIFACT, "latest")
classifier = load_classifier_from_path(path, device=device).eval()
print(f"classifier: {resolved} | heads: {classifier.config.names}")

dataset = build_dataset(DATASET, train=False)
loader = DataLoader(dataset, batch_size=512, shuffle=False)
print(f"{DATASET} val: {len(dataset)} images")

## Predict the whole split

In [ ]:
images, labels, predictions = [], [], []
with torch.no_grad():
    for batch_images, batch_labels in loader:
        images.append(batch_images)
        labels.append(batch_labels.long())
        predictions.append(classifier.predict(batch_images.to(device)).cpu())
images, labels, predictions = torch.cat(images), torch.cat(labels), torch.cat(predictions)

correct = predictions == labels
for i, name in enumerate(classifier.config.names):
    print(f"{name:<6} accuracy {correct[:, i].float().mean():.4f}")
print(f"all three correct {correct.all(dim=1).float().mean():.4f}")

In [ ]:
def describe(label: torch.Tensor) -> str:
    digit, fg, bg = label.tolist()
    return f"{digit} {FG_NAMES[fg]}/{BG_NAMES[bg]}"


def show(indices: torch.Tensor, title: str):
    captions = [
        f"true {describe(labels[i])}\npred {describe(predictions[i])}" for i in indices
    ]
    colours = ["black" if correct[i].all() else "red" for i in indices]
    return plot_captioned_images(
        images[indices], captions, title=title, caption_colours=colours
    )

## A random sample

In [ ]:
N_SHOWN = 32

random_indices = torch.randperm(len(images))[:N_SHOWN]
figure = show(random_indices, f"{N_SHOWN} random val images")

## Misclassifications

In [ ]:
wrong = (~correct.all(dim=1)).nonzero(as_tuple=True)[0]
print(f"{len(wrong)} of {len(images)} misclassified in at least one factor")
if len(wrong):
    wrong_indices = wrong[torch.randperm(len(wrong))[:N_SHOWN]]
    figure = show(wrong_indices, f"{len(wrong_indices)} of the {len(wrong)} misclassified")